In [3]:
# Model settings
WHISPER_MODEL = "small"          # tiny, base, small, medium, large-v3
LANGUAGE = "en"
FP16 = False                     # False for macOS CPU

# Wake word
TRIGGER_PHRASE = "hey deepgi"
TRIGGER_VARIANTS = [
    "hey deepgi", "hey deep gi", "hey dgi", "hey d gi",
    "hey deepgee", "hey deep gee", "hey travis", "hey davis", "hey jarvis"
]
FUZZY_THRESHOLD = 0.7

# Audio settings
SAMPLE_RATE = 16000
TRIGGER_CHUNK_DURATION = 3       # seconds to listen for trigger
FINDING_DURATION = 8             # seconds to record after trigger

# Medical vocabulary hint for Whisper
INITIAL_PROMPT = """DeepGI colonoscopy findings: polyp, lesion, bleeding,
diverticulum, sigmoid colon, cecum, rectum, ascending colon, descending colon,
transverse colon, hepatic flexure, splenic flexure, Paris classification,
Boston Bowel Score, pedunculated, sessile, flat, biopsy, resection"""

# Training mode toggle
USE_FINETUNED_ASR = False        # True to use fine-tuned model
USE_FINETUNED_VAD = False        # True to use trained wake word model
FINETUNED_ASR_PATH = "outputs/models/whisper-deepgi"
FINETUNED_VAD_PATH = "outputs/"


In [4]:
"""
Train a wake word classifier using MFCC features + CNN.

Approach:
  1. Load all .wav files from metadata.csv
  2. Convert audio to fixed length
  3. Extract MFCC as 2D feature: (1, n_mfcc, time)
  4. Train CNN binary classifier
  5. Validate with train/val split
  6. Use original BCEWithLogitsLoss(pos_weight=...)
  7. Add gradient clipping + scheduler
  8. Save best model by validation loss
  9. Save history and plot graphs
"""

import csv
import os
import sys
import json
import random
import copy

import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torchaudio.transforms as T
import torchaudio.functional as AF


# =========================
# Config
# =========================

try:
    import config

    SAMPLE_RATE = getattr(config, "SAMPLE_RATE", 16000)
    OUTPUT_MODEL_DIR = getattr(
        config,
        "FINETUNED_VAD_PATH",
        "/content/drive/MyDrive/outputs/models/vad-deepgi",
    )
except Exception:
    SAMPLE_RATE = 16000
    OUTPUT_MODEL_DIR = "/content/outputs"


# metadata ที่ path ข้างในเป็น:
# /content/drive/MyDrive/dataset/sample_0001.wav
METADATA_CSV = "/content/metadata(1)(1).csv"

# ถ้า metadata อยู่ใน Google Drive ให้ใช้บรรทัดนี้แทน
# METADATA_CSV = "/content/drive/MyDrive/dataset/metadata_drive_dataset_no_audio_vad.csv"


TRIGGER_KEYWORDS = [
    "hey deepgi",
    "hey deep gi",
    "hey dgi",
    "hey deepgee",
    "hey deep gee",
    "เฮ้ deepgi",
    "เฮ deepgi",
    "เฮ้ deep gi",
    "เฮ้ dgi",
]

THRESHOLD = 0.80

CLIP_SECONDS = 3.0
N_SAMPLES = int(SAMPLE_RATE * CLIP_SECONDS)

BATCH_SIZE = 32
EPOCHS = 120
LR = 4e-5
WEIGHT_DECAY = 1e-4

VAL_RATIO = 0.20
SEED = 42

GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# MFCC
# =========================

_MFCC = T.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=40,
    melkwargs={
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 80,
    },
)


# =========================
# Utils
# =========================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_path(path: str) -> str:
    return path.replace("\\", "/")


def load_audio(filepath: str) -> torch.Tensor:
    rate, data = wav.read(filepath)

    # stereo -> mono
    if data.ndim > 1:
        data = data.mean(axis=1)

    original_dtype = data.dtype
    audio = data.astype(np.float32)

    # normalize int audio
    if np.issubdtype(original_dtype, np.integer):
        max_value = np.iinfo(original_dtype).max
        audio = audio / max_value

    tensor = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)
    # shape: (1, samples)

    # resample ถ้า sample rate ไม่ตรง
    if rate != SAMPLE_RATE:
        tensor = AF.resample(tensor, rate, SAMPLE_RATE)

    # pad / trim ให้เท่ากับ CLIP_SECONDS
    num_samples = tensor.shape[-1]

    if num_samples < N_SAMPLES:
        pad_amount = N_SAMPLES - num_samples
        tensor = F.pad(tensor, (0, pad_amount))
    else:
        tensor = tensor[:, :N_SAMPLES]

    return tensor


def extract_mfcc(filepath: str) -> torch.Tensor:
    audio = load_audio(filepath)
    # shape: (1, samples)

    mfcc = _MFCC(audio)
    # shape: (1, 40, time)

    mfcc = (mfcc - mfcc.mean()) / (mfcc.std() + 1e-6)

    return mfcc.float()
    # shape: (1, 40, time)


# =========================
# Dataset
# =========================

class WakeWordDataset(Dataset):
    def __init__(self, metadata_csv: str):
        self.samples = []

        if not os.path.exists(metadata_csv):
            print(f"[ERROR] Metadata CSV not found: {metadata_csv}")
            sys.exit(1)

        missing_count = 0

        with open(metadata_csv, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            required_cols = {"audio_filepath", "text", "label"}
            fieldnames = set(reader.fieldnames or [])

            missing_cols = required_cols - fieldnames
            if missing_cols:
                raise ValueError(f"Missing columns in CSV: {missing_cols}")

            for row in reader:
                path = normalize_path(row["audio_filepath"])
                label = int(row["label"])

                if not os.path.exists(path):
                    print(f"[WARN] Missing file: {path} — skipping")
                    missing_count += 1
                    continue

                self.samples.append((path, label))

        print(f"[DATA] Loaded usable samples: {len(self.samples)}")
        print(f"[DATA] Missing files skipped: {missing_count}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        mfcc = extract_mfcc(path)
        label = torch.tensor(label, dtype=torch.float32)

        return mfcc, label


# =========================
# CNN model: มองภาพใหญ่ขึ้น
# =========================

class WakeWordCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            # input: (batch, 1, 40, time)

            nn.Conv2d(1, 32, kernel_size=(5, 9), padding=(2, 4)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            nn.Conv2d(32, 64, kernel_size=(5, 9), padding=(2, 4)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            # dilation ช่วยให้มองตามแกนเวลากว้างขึ้น
            nn.Conv2d(
                64,
                128,
                kernel_size=(3, 7),
                padding=(1, 6),
                dilation=(1, 2),
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(
                128,
                128,
                kernel_size=(3, 7),
                padding=(2, 6),
                dilation=(2, 2),
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Dropout2d(0.20),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        # x shape: (batch, 1, 40, time)
        x = self.net(x)
        x = x.flatten(1)
        logits = self.classifier(x).squeeze(1)
        return logits


# =========================
# Train / Val split
# =========================

def make_stratified_split(dataset, val_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)

    pos_indices = []
    neg_indices = []

    for i, (_, label) in enumerate(dataset.samples):
        if label == 1:
            pos_indices.append(i)
        else:
            neg_indices.append(i)

    pos_indices = np.array(pos_indices)
    neg_indices = np.array(neg_indices)

    rng.shuffle(pos_indices)
    rng.shuffle(neg_indices)

    def val_count(n):
        if n <= 1:
            return 0
        return min(max(1, int(round(n * val_ratio))), n - 1)

    n_val_pos = val_count(len(pos_indices))
    n_val_neg = val_count(len(neg_indices))

    val_indices = np.concatenate([
        pos_indices[:n_val_pos],
        neg_indices[:n_val_neg],
    ])

    train_indices = np.concatenate([
        pos_indices[n_val_pos:],
        neg_indices[n_val_neg:],
    ])

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)

    return train_indices.tolist(), val_indices.tolist()


# =========================
# Metrics
# =========================

def binary_metrics_from_logits(logits, labels, threshold=0.80):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    labels = labels.float()

    tp = ((preds == 1) & (labels == 1)).sum().item()
    tn = ((preds == 0) & (labels == 0)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()

    total = tp + tn + fp + fn

    acc = (tp + tn) / max(total, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    return {
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_count = 0

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            total_count += y.numel()

            all_logits.append(logits.detach().cpu())
            all_labels.append(y.detach().cpu())

    avg_loss = total_loss / max(total_count, 1)

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    metrics_50 = binary_metrics_from_logits(
        all_logits,
        all_labels,
        threshold=0.50,
    )

    metrics_80 = binary_metrics_from_logits(
        all_logits,
        all_labels,
        threshold=THRESHOLD,
    )

    return {
        "loss": avg_loss,
        "metrics_50": metrics_50,
        "metrics_80": metrics_80,
    }


# =========================
# Plot
# =========================

def plot_history(history, output_dir):
    epochs = range(1, len(history["train_loss"]) + 1)

    os.makedirs(output_dir, exist_ok=True)

    # Loss graph
    plt.figure()
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Train vs Validation Loss")
    plt.legend()
    plt.grid(True)
    loss_plot_path = os.path.join(output_dir, "loss_curve.png")
    plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
    plt.close()

    # Accuracy @ 0.50
    plt.figure()
    plt.plot(epochs, history["train_acc_50"], label="Train Acc @ 0.50")
    plt.plot(epochs, history["val_acc_50"], label="Val Acc @ 0.50")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Train vs Validation Accuracy @ 0.50")
    plt.legend()
    plt.grid(True)
    acc50_plot_path = os.path.join(output_dir, "accuracy_050_curve.png")
    plt.savefig(acc50_plot_path, dpi=150, bbox_inches="tight")
    plt.close()

    # Accuracy @ THRESHOLD
    plt.figure()
    plt.plot(epochs, history["train_acc_80"], label=f"Train Acc @ {THRESHOLD:.2f}")
    plt.plot(epochs, history["val_acc_80"], label=f"Val Acc @ {THRESHOLD:.2f}")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Train vs Validation Accuracy @ {THRESHOLD:.2f}")
    plt.legend()
    plt.grid(True)
    acc80_plot_path = os.path.join(output_dir, "accuracy_080_curve.png")
    plt.savefig(acc80_plot_path, dpi=150, bbox_inches="tight")
    plt.close()

    # FP graph
    plt.figure()
    plt.plot(epochs, history["train_fp_80"], label=f"Train FP @ {THRESHOLD:.2f}")
    plt.plot(epochs, history["val_fp_80"], label=f"Val FP @ {THRESHOLD:.2f}")
    plt.xlabel("Epoch")
    plt.ylabel("False Positive Count")
    plt.title(f"False Positive Count @ {THRESHOLD:.2f}")
    plt.legend()
    plt.grid(True)
    fp_plot_path = os.path.join(output_dir, "false_positive_curve.png")
    plt.savefig(fp_plot_path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"[PLOT] Loss plot saved to: {loss_plot_path}")
    print(f"[PLOT] Acc @ 0.50 plot saved to: {acc50_plot_path}")
    print(f"[PLOT] Acc @ {THRESHOLD:.2f} plot saved to: {acc80_plot_path}")
    print(f"[PLOT] FP plot saved to: {fp_plot_path}")


# =========================
# Main
# =========================

def main():
    set_seed(SEED)

    print(f"[INFO] DEVICE = {DEVICE}")
    print(f"[INFO] SAMPLE_RATE = {SAMPLE_RATE}")
    print(f"[INFO] METADATA_CSV = {METADATA_CSV}")
    print(f"[INFO] OUTPUT_MODEL_DIR = {OUTPUT_MODEL_DIR}")

    print("\n[DATA] Loading samples...")
    dataset = WakeWordDataset(METADATA_CSV)

    labels = [label for _, label in dataset.samples]
    n_pos = sum(labels)
    n_neg = len(labels) - n_pos

    print(f"[DATA] Positive trigger samples: {n_pos}")
    print(f"[DATA] Negative samples: {n_neg}")

    if n_pos == 0:
        print("[ERROR] No trigger samples found in metadata.csv.")
        sys.exit(1)

    if n_neg == 0:
        print("[ERROR] No non-trigger samples found. Need both classes to train.")
        sys.exit(1)

    train_indices, val_indices = make_stratified_split(
        dataset,
        val_ratio=VAL_RATIO,
        seed=SEED,
    )

    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)

    print(f"[DATA] Train samples: {len(train_dataset)}")
    print(f"[DATA] Val samples: {len(val_dataset)}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )

    model = WakeWordCNN().to(DEVICE)

    # =========================
    # Loss แบบเดิมตอนแรก
    # =========================
    pos_weight = torch.tensor(
        [n_neg / max(n_pos, 1)],
        device=DEVICE,
        dtype=torch.float32,
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    print(f"[LOSS] BCEWithLogitsLoss pos_weight = {pos_weight.item():.4f}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
    )

    history = {
        "train_loss": [],
        "val_loss": [],

        "train_acc_50": [],
        "val_acc_50": [],

        "train_acc_80": [],
        "val_acc_80": [],

        "train_precision_80": [],
        "val_precision_80": [],

        "train_recall_80": [],
        "val_recall_80": [],

        "train_f1_80": [],
        "val_f1_80": [],

        "train_fp_80": [],
        "val_fp_80": [],

        "train_fn_80": [],
        "val_fn_80": [],

        "lr": [],
    }

    os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

    best_val_loss = float("inf")
    best_epoch = -1
    best_model_state = None
    no_improve_count = 0

    print(f"\n[TRAIN] Training on {DEVICE}...")

    for epoch in range(1, EPOCHS + 1):
        model.train()

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()

            # ช่วยลด spike ของ loss
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRAD_CLIP_NORM,
            )

            optimizer.step()

        train_result = evaluate(
            model,
            train_loader,
            criterion,
        )

        val_result = evaluate(
            model,
            val_loader,
            criterion,
        )

        train_loss = train_result["loss"]
        val_loss = val_result["loss"]

        train_m50 = train_result["metrics_50"]
        val_m50 = val_result["metrics_50"]

        train_m80 = train_result["metrics_80"]
        val_m80 = val_result["metrics_80"]

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        history["train_acc_50"].append(train_m50["acc"])
        history["val_acc_50"].append(val_m50["acc"])

        history["train_acc_80"].append(train_m80["acc"])
        history["val_acc_80"].append(val_m80["acc"])

        history["train_precision_80"].append(train_m80["precision"])
        history["val_precision_80"].append(val_m80["precision"])

        history["train_recall_80"].append(train_m80["recall"])
        history["val_recall_80"].append(val_m80["recall"])

        history["train_f1_80"].append(train_m80["f1"])
        history["val_f1_80"].append(val_m80["f1"])

        history["train_fp_80"].append(train_m80["fp"])
        history["val_fp_80"].append(val_m80["fp"])

        history["train_fn_80"].append(train_m80["fn"])
        history["val_fn_80"].append(val_m80["fn"])

        history["lr"].append(current_lr)

        improved = val_loss < best_val_loss

        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improve_count = 0
            best_model_state = copy.deepcopy(model.state_dict())

            model_path = os.path.join(OUTPUT_MODEL_DIR, "classifier_cnn.pt")

            torch.save(
                {
                    "model_state_dict": best_model_state,
                    "threshold": THRESHOLD,
                    "sample_rate": SAMPLE_RATE,
                    "clip_seconds": CLIP_SECONDS,
                    "n_mfcc": 40,
                    "trigger_keywords": TRIGGER_KEYWORDS,
                    "val_ratio": VAL_RATIO,
                    "best_val_loss": best_val_loss,
                    "best_epoch": best_epoch,
                    "architecture": "WakeWordCNN_large_receptive_field",
                    "loss": {
                        "type": "BCEWithLogitsLoss",
                        "pos_weight": pos_weight.item(),
                    },
                    "optimizer": {
                        "type": "AdamW",
                        "lr": LR,
                        "weight_decay": WEIGHT_DECAY,
                    },
                },
                model_path,
            )
        else:
            no_improve_count += 1

        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"lr={current_lr:.2e} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"train_acc@0.50={train_m50['acc']:.1%} | "
            f"val_acc@0.50={val_m50['acc']:.1%} | "
            f"train_acc@{THRESHOLD:.2f}={train_m80['acc']:.1%} | "
            f"val_acc@{THRESHOLD:.2f}={val_m80['acc']:.1%} | "
            f"val_precision={val_m80['precision']:.1%} | "
            f"val_recall={val_m80['recall']:.1%} | "
            f"val_fp={val_m80['fp']} | "
            f"val_fn={val_m80['fn']} | "
            f"{'BEST' if improved else ''}"
        )

        if no_improve_count >= EARLY_STOP_PATIENCE:
            print(
                f"\n[EARLY STOP] No improvement for {EARLY_STOP_PATIENCE} epochs. "
                f"Best epoch = {best_epoch}, best_val_loss = {best_val_loss:.4f}"
            )
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    history_path = os.path.join(OUTPUT_MODEL_DIR, "training_history.json")

    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)

    plot_history(history, OUTPUT_MODEL_DIR)

    final_model_path = os.path.join(OUTPUT_MODEL_DIR, "classifier_cnn.pt")

    print("\n[DONE] Training complete.")
    print(f"[DONE] Best epoch: {best_epoch}")
    print(f"[DONE] Best val loss: {best_val_loss:.4f}")
    print(f"[DONE] Model saved to: {final_model_path}")
    print(f"[DONE] History saved to: {history_path}")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install -q matplotlib scipy torch torchaudio openai-whisper sounddevice \
    transformers datasets accelerate evaluate jiwer \
    openwakeword kokoro soundfile webrtcvad-wheels

In [ ]:
# ================================
# Test evaluation only
# Load model -> Predict -> Confusion Matrix
# ================================

import csv
import os
import sys
import json

import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T
import torchaudio.functional as AF


# -------- Config --------
TEST_METADATA_CSV = "/content/metadata_validate_path.csv"

# ใช้ path จาก config ด้านบน ถ้ามี ไม่งั้นใช้ default
OUTPUT_MODEL_DIR = globals().get("FINETUNED_VAD_PATH", "outputs/")
MODEL_PATH = os.path.join(OUTPUT_MODEL_DIR, "classifier_cnn.pt")

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -------- Model architecture ต้องตรงกับตอน train --------
class WakeWordCNNForTest(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            # input: (batch, 1, 40, time)

            # มองกว้างขึ้นจากเดิม 3x5 เป็น 5x9
            nn.Conv2d(1, 32, kernel_size=(5, 9), padding=(2, 4)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            # มอง pattern ใหญ่ขึ้นอีก
            nn.Conv2d(32, 64, kernel_size=(5, 9), padding=(2, 4)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            # ใช้ dilation ให้มองแกนเวลาไกลขึ้น
            # kernel จริงเหมือนมองประมาณ 3 x 13
            nn.Conv2d(
                64,
                128,
                kernel_size=(3, 7),
                padding=(1, 6),
                dilation=(1, 2),
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # มองทั้ง frequency และ time กว้างขึ้นอีก
            # kernel จริงประมาณ 5 x 13
            nn.Conv2d(
                128,
                128,
                kernel_size=(3, 7),
                padding=(2, 6),
                dilation=(2, 2),
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Dropout2d(0.20),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        # x shape: (batch, 1, 40, time)
        x = self.net(x)
        x = x.flatten(1)
        logits = self.classifier(x).squeeze(1)
        return logits


# -------- Load checkpoint --------
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

# รองรับ checkpoint แบบ dict จาก train code
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
    THRESHOLD_TEST = 0.65
    SAMPLE_RATE_TEST = checkpoint.get("sample_rate", globals().get("SAMPLE_RATE", 16000))
    CLIP_SECONDS_TEST = checkpoint.get("clip_seconds", 3.0)
    N_MFCC_TEST = checkpoint.get("n_mfcc", 40)
else:
    # fallback กรณี save มาเป็น state_dict ตรง ๆ
    state_dict = checkpoint
    THRESHOLD_TEST = 0.70
    SAMPLE_RATE_TEST = globals().get("SAMPLE_RATE", 16000)
    CLIP_SECONDS_TEST = 3.0
    N_MFCC_TEST = 40

N_SAMPLES_TEST = int(SAMPLE_RATE_TEST * CLIP_SECONDS_TEST)

print(f"Loaded model: {MODEL_PATH}")
print(f"Device: {DEVICE}")
print(f"Threshold: {THRESHOLD_TEST}")
print(f"Sample rate: {SAMPLE_RATE_TEST}")
print(f"Clip seconds: {CLIP_SECONDS_TEST}")
print(f"n_mfcc: {N_MFCC_TEST}")


# -------- MFCC extractor --------
_MFCC_TEST = T.MFCC(
    sample_rate=SAMPLE_RATE_TEST,
    n_mfcc=N_MFCC_TEST,
    melkwargs={
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 80,
    },
)


def load_audio_for_test(filepath: str) -> torch.Tensor:
    rate, data = wav.read(filepath)

    # stereo -> mono
    if data.ndim > 1:
        data = data.mean(axis=1)

    audio = data.astype(np.float32)

    # normalize int audio
    if np.issubdtype(data.dtype, np.integer):
        max_value = np.iinfo(data.dtype).max
        audio = audio / max_value

    tensor = torch.tensor(audio).unsqueeze(0)  # (1, samples)

    # resample ถ้า sample rate ไม่ตรง
    if rate != SAMPLE_RATE_TEST:
        tensor = AF.resample(tensor, rate, SAMPLE_RATE_TEST)

    # pad / trim ให้ยาวเท่ากัน
    num_samples = tensor.shape[-1]

    if num_samples < N_SAMPLES_TEST:
        pad_amount = N_SAMPLES_TEST - num_samples
        tensor = torch.nn.functional.pad(tensor, (0, pad_amount))
    else:
        tensor = tensor[:, :N_SAMPLES_TEST]

    return tensor


def extract_mfcc_for_test(filepath: str) -> torch.Tensor:
    audio = load_audio_for_test(filepath)
    mfcc = _MFCC_TEST(audio)  # (1, n_mfcc, time)

    # normalize ต่อคลิปเหมือนตอน train
    mfcc = (mfcc - mfcc.mean()) / (mfcc.std() + 1e-6)

    return mfcc.float()


# -------- Test Dataset --------
class WakeWordTestDataset(Dataset):
    def __init__(self, metadata_csv: str):
        self.samples = []

        if not os.path.exists(metadata_csv):
            raise FileNotFoundError(f"Test metadata not found: {metadata_csv}")

        with open(metadata_csv, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            for row in reader:
                path = row["audio_filepath"]
                label = int(row["label"])

                if not os.path.exists(path):
                    print(f"[WARN] Missing file: {path} — skipping")
                    continue

                self.samples.append((path, label))

        if len(self.samples) == 0:
            raise RuntimeError("No valid test samples found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        x = extract_mfcc_for_test(path)
        y = torch.tensor(label).float()
        return x, y, path


# -------- Metrics --------
def compute_binary_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def plot_confusion_matrix(metrics):
    cm = np.array([
        [metrics["tn"], metrics["fp"]],
        [metrics["fn"], metrics["tp"]],
    ])

    class_names = ["non-trigger", "trigger"]

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(f"Confusion Matrix @ threshold {THRESHOLD_TEST}")
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=30)
    plt.yticks(tick_marks, class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()


# -------- Predict + Evaluate --------
test_dataset = WakeWordTestDataset(TEST_METADATA_CSV)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

labels = [label for _, label in test_dataset.samples]
print("\nTest dataset")
print(f"  total samples : {len(test_dataset)}")
print(f"  trigger       : {sum(labels)}")
print(f"  non-trigger   : {len(labels) - sum(labels)}")

model = WakeWordCNNForTest().to(DEVICE)
model.load_state_dict(state_dict)
model.eval()

y_true = []
y_pred = []
y_prob = []
paths = []

with torch.no_grad():
    for x, y, batch_paths in test_loader:
        x = x.to(DEVICE)

        logits = model(x)
        probs = torch.sigmoid(logits)
        preds = (probs >= THRESHOLD_TEST).int()

        y_true.extend(y.numpy().astype(int).tolist())
        y_pred.extend(preds.cpu().numpy().astype(int).tolist())
        y_prob.extend(probs.cpu().numpy().tolist())
        paths.extend(batch_paths)

metrics = compute_binary_metrics(y_true, y_pred)

print("\nEvaluation Result")
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall   : {metrics['recall']:.4f}")
print(f"F1-score : {metrics['f1']:.4f}")

print("\nConfusion Matrix")
print("Rows = True label, Columns = Predicted label")
print("           pred_0  pred_1")
print(f"true_0     {metrics['tn']:6d}  {metrics['fp']:6d}")
print(f"true_1     {metrics['fn']:6d}  {metrics['tp']:6d}")

print("\nMeaning")
print(f"TN = {metrics['tn']} | non-trigger -> predicted non-trigger")
print(f"FP = {metrics['fp']} | non-trigger -> predicted trigger")
print(f"FN = {metrics['fn']} | trigger -> predicted non-trigger")
print(f"TP = {metrics['tp']} | trigger -> predicted trigger")

plot_confusion_matrix(metrics)


# -------- Optional: save predictions --------
prediction_csv = os.path.join(OUTPUT_MODEL_DIR, "test_predictions.csv")
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

with open(prediction_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["audio_filepath", "true_label", "pred_label", "probability"])

    for path, true_label, pred_label, prob in zip(paths, y_true, y_pred, y_prob):
        writer.writerow([path, true_label, pred_label, prob])

print(f"\nSaved predictions to: {prediction_csv}")
